In [1]:
pip install pandas pyarrow datasets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from datasets import load_dataset

In [4]:
import pandas as pd
import glob

# Step 1: Set the folder path (Downloads in your case)
folder_path = "C:/Users/manjo/Downloads/"

# Step 2: Use glob to get all 10 parquet files
files = glob.glob(folder_path + "000*.parquet")

# Step 3: Load and merge all the parquet files into one DataFrame
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

# Step 4: Print basic info to verify
print(f"Total records: {len(df)}")
print(df.head())

Total records: 9100
                                               image  label
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      0
1  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      0
2  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      0
3  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      0
4  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      0


In [5]:
!pip install pytesseract

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from PIL import Image
import pytesseract
import io

# Set Tesseract path
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [8]:
# Function to extract text from raw image bytes
def extract_text(row):
    try:
        img_bytes = row['image']['bytes']  # Access image bytes
        image = Image.open(io.BytesIO(img_bytes))  # Convert bytes to PIL image
        text = pytesseract.image_to_string(image)  # OCR to extract text
        return text
    except Exception as e:
        return f"[Error: {e}]"

In [10]:
from PIL import Image
import pytesseract
import io

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # Adjust if needed

def extract_text(row):
    try:
        img_bytes = row['image']['bytes']
        image = Image.open(io.BytesIO(img_bytes))
        image = image.resize((image.width // 2, image.height // 2))  # Optional: resize for speed
        return pytesseract.image_to_string(image)
    except Exception as e:
        return f"[Error: {e}]"



In [15]:
df['ocr_text'] = df.apply(extract_text, axis=1)

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
documents = df['ocr_text'].tolist()
embeddings = model.encode(documents)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manjo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
documents = df['ocr_text'].tolist()  # OCR text column
embeddings = model.encode(documents, show_progress_bar=True)  # Generate embeddings

Batches:   0%|          | 0/285 [00:00<?, ?it/s]

In [18]:
import numpy as np
np.save("ocr_text_embeddings.npy", embeddings)

In [19]:
df['embedding'] = embeddings.tolist()
df.to_parquet("ocr_text_with_embeddings.parquet", index=False)

In [24]:
# Join all OCR text into one single string
full_text = "\n\n".join(df['ocr_text'].dropna().tolist())

# Save it to a .txt file
with open("extracted_texts.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [25]:
from IPython.display import FileLink
FileLink("extracted_texts.txt")

C:\Users\manjo\extracted_texts.txt

In [26]:
# Define common OCR fixes
corrections = {
    "Wixxrepia": "Wikipedia",
    "Creale": "Create",
    "Donale": "Donate",
    "Adicle": "Article",
    "Viewhistory": "View history"
}

# Apply corrections to the text
full_text = "\n\n".join(df['ocr_text'].dropna().tolist())
for wrong, right in corrections.items():
    full_text = full_text.replace(wrong, right)

# Save corrected text
with open("extracted_texts_corrected.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [27]:
from IPython.display import FileLink
FileLink("extracted_texts_corrected.txt")

C:\Users\manjo\extracted_texts_corrected.txt

In [28]:
import re

# Original OCR text
full_text = "\n\n".join(df['ocr_text'].dropna().tolist())

# Define flexible patterns and fixes
fix_patterns = {
    r"\bW[ilx]{1,2}[kq]{0,1}p[aei]{0,2}d{0,1}i{0,2}a\b": "Wikipedia",  # various spellings
    r"\bDonale\b": "Donate",
    r"\bCreale\b": "Create",
    r"\bAdicle\b": "Article",
    r"\bViewhistory\b": "View history"
}

# Apply regex replacements
for pattern, correct in fix_patterns.items():
    full_text = re.sub(pattern, correct, full_text, flags=re.IGNORECASE)

# Save corrected text
with open("extracted_texts_corrected.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [29]:
import numpy as np

# Load the embeddings
embeddings = np.load("ocr_text_embeddings.npy")

# Check the shape
print(embeddings.shape)   # e.g. (9100, 384)

# View the first embedding (vector of numbers)
print(embeddings[0])

(9100, 384)
[-1.18833676e-01 -7.85546228e-02  3.50314043e-02 -3.50957550e-02
 -8.00016150e-02  8.95897150e-02 -3.03262123e-03  1.37477115e-01
  1.97609309e-02 -2.54979711e-02  5.36187738e-03 -1.35159686e-01
  7.28047788e-02 -4.22027074e-02 -1.15149876e-03  3.83399054e-02
  3.35396305e-02  2.43818052e-02 -5.40605672e-02  1.02149742e-02
 -6.73814714e-02 -6.53596148e-02  4.71921191e-02 -3.32746451e-04
 -1.59842856e-02  1.57257766e-02  2.89444602e-03  4.60340735e-03
 -1.91641264e-02  6.01257244e-03  5.45206070e-02  2.97227968e-02
 -3.81193534e-02  3.41134630e-02 -1.32901210e-03  4.15382758e-02
 -2.05383971e-02 -2.67775543e-02 -5.20601347e-02 -1.30613297e-02
 -2.61555389e-02 -1.68193858e-02 -5.11852279e-02 -8.17209035e-02
 -7.23908795e-03 -1.30603805e-01 -2.44277455e-02  4.80771251e-03
 -4.14452665e-02 -3.32110971e-02 -4.85338569e-02 -3.09121218e-02
  1.86374877e-02 -2.06957255e-02 -5.67965256e-03  5.53015387e-03
  9.73994434e-02  1.92254689e-02  9.60310474e-02  1.45433601e-02
 -4.36183698e

In [30]:
from IPython.display import FileLink
FileLink("ocr_text_embeddings.npy")

C:\Users\manjo\ocr_text_embeddings.npy

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

with open("extracted_texts_corrected.txt", "r", encoding="utf-8") as f:
    documents = f.read().split("\n\n")  # Split by double line breaks

embeddings = model.encode(documents, show_progress_bar=True)

np.save("final_embeddings.npy", embeddings)

Batches:   0%|          | 0/5015 [00:00<?, ?it/s]